# **OSSE subsampling exercise within SOCOMv2**
Author: Daniel J. Ford (d.ford@exeter.ac.uk)

Version: v1

Date: 15/05/2026

Other contacts: Amanda Fay and Thea Heimdal Hatlen

This code calculates the air-sea CO2 fluxes for the model truth, and then adds the additional data product fields into the files and calculates. These are then combined into one large file for each model (i.e model truth, with all the OSSE experiments for each product).

# OceanICU framework
This code requires the OceanICU framework code developed at Exeter University for the data wrangling, flux calculations (with FluxEngine) and the intergration into the global ocean CO2 sink estimates. This code cannot be installed via pip or conda at this time, and must be downloaded and then put in folder for the code to then import.
The code can be downloaded from Github here: [https://github.com/JamieLab/OceanICU](https://github.com/JamieLab/OceanICU)

This code should be unzip and the location of this folder be place in the variable in the next block. 

In [1]:
OceanICU_framework_loc = 'C:/Users/df391/OneDrive - University of Exeter/Post_Doc_ESA_Contract/OceanICU'

## Importing the packages needed
The packages needed are imported here. The environment provided with the OceanICU framework includes all the packages needed and that is the easiest way to get started. This can be installed by creating a conda environment from the environment.yml in the OceanICU code base.

The working directory will include a folder with the truth (i.e the truth folder on Keeper), along with a "Output" folder which will contain the netCDF output from each provider. 

Note: The IPSL naming needs to changes from 'IPSL_NEMO_PICES' to 'IPSL_NEMO_PISCES'. The individual data product file naming convention also needs to be checked as some groups have minor formating issues that will break the script.

In [2]:
import sys
import os
import datetime
from netCDF4 import Dataset
import numpy as np
import glob

# These add the OceanICU framwork locations to the system path so we can then import
# the scripts and functions
sys.path.append(OceanICU_framework_loc)
sys.path.append(os.path.join(OceanICU_framework_loc,'Data_Loading'))
import data_utils as du

working_directory = 'D:/OSSE_Experiments' # Top level working directory for the OSSE analysis
truth_directory = os.path.join(working_directory,'Truth') # Directory for the model truth data (from Keeper)
data_prod_directory = os.path.join(working_directory,'Output') # Directory for all the data product outputs (from Keeper)

code_path = os.getcwd()
flux_config = os.path.join(code_path,'SOCOMv2_flux_config.conf') # SOCOMv2 FluxEngine config for ERA5 winds scaled to 14C constraint
print(flux_config)
version = 'v1' # Version of the output files (for version controlling)

copts= {"zlib":True,}
# This is the dictionary for the models being used. The main entry is the model name with underscores (i.e for folder directory)
# and the entry is the same with '-' for the data product file structure.
models = {'FESOM2_REcoM': 'FESOM2-REcoM',
       'IPSL_NEMO_PISCES': 'IPSL-NEMO-PISCES',
       'MRI_ESM2': 'MRI-ESM2'}

start_yr = 1980 # Start year for the models
end_yr = 2024 # End year for the models
log,lag = du.reg_grid(lat=1,lon=1) # Sets up a 1 degree grid (-180 to 180, -90 to 90).

###
raw_wind = "F:/Data/ERA5/DAILY/monthly"
raw_pressure = "F:/Data/ERA5/MONTHLY/DATA"
area_file = os.path.join(working_directory,'OSSE_ocean_area_mask_-180_180_v1.nc')
reccap_file = os.path.join(working_directory,'RECCAP2_region_masks_all_v20221025.nc')

C:\Users\df391\OneDrive - University of Exeter\Post_Doc_ESA_Contract\SOCOMv2\SOCOMv2\Experiment_OSSE\SOCOMv2_flux_config.conf


## Retrieving the ERA5 datasets for wind and atmospheric pressure
I start this script assuming the raw data has been downloaded from ERA5, these just average the data from 0.25 degree to 1 degree. 
If you haven't contact Daniel J. Ford for a script to download the ERA5 data. The raw hourly wind data is ~1TB and takes a long time to download.

I have to %%capture this code block as the outputs cause the web side of Jupyter Notebook to crash due to the amount of console output.

In [3]:
%%capture
du.makefolder(os.path.join(working_directory,'wind'))
du.makefolder(os.path.join(working_directory,'wind','ws'))
du.makefolder(os.path.join(working_directory,'wind','ws2'))
du.makefolder(os.path.join(working_directory,'pressure'))

from Data_Loading.ERA5_data_download import era5_average
era5_average(loc = raw_wind, outloc=os.path.join(working_directory,'wind','ws'),log=log,lag=lag,var='ws',start_yr = start_yr,end_yr =end_yr)
era5_average(loc = raw_wind, outloc=os.path.join(working_directory,'wind','ws2'),log=log,lag=lag,var='ws2',start_yr = start_yr,end_yr =end_yr)
era5_average(loc = raw_pressure, outloc=os.path.join(working_directory,'pressure'),log=log,lag=lag,var='msl',start_yr = start_yr,end_yr =end_yr)

## Setting up the flux calculations for the model truth
Here we setup the base files that contain the information needed for the 'truth' flux calculations. We then run the flux calculations with FluxEngine, and then we append the flux calculation information back into the working file (with the attributes added). This section for three models takes ~30 mins.

I have to %%capture this code block as the outputs cause the web side of Jupyter Notebook to crash due to the amount of console output.

In [3]:
%%capture
import construct_input_netcdf as cinp
from neural_network_train import make_save_tree
import fluxengine_driver as fl
from fluxengine.core import fe_setup_tools as fluxengine


for model in list(models.keys()):
    # Setup the model location directory and making the save tree that the OceanICU framework expects.
    # Setup the name of data file in the correct place
    model_location = os.path.join(truth_directory,model)
    # make_save_tree(model_location)
    data_file = os.path.join(model_location,'OSSE_'+model+'_'+version+'.nc')

    # This is the directory structure and locations of the variables to input into the
    # flux calculation.
    #Vars should have each entry as [Extra_Name, netcdf_variable_name,data_location,produce_anomaly]
    vars = [['model','tos',os.path.join(model_location,'tos','%Y_%m*.nc'),0], # Temperature
        ['model','sos',os.path.join(model_location,'sos','%Y_%m*.nc'),0], # Salinity
        ['model','fice',os.path.join(model_location,'fice','%Y_%m*.nc'),0], # Sea Ice
        ['model','sfco2',os.path.join(model_location,'sfco2','%Y_%m*.nc'),0], # Model fCO2sw fields
        ['model','xco2',os.path.join(model_location,'xco2','%Y_%m*.nc'),0],# Atmospheric xCO2
        ['ERA5','ws',os.path.join(working_directory,'wind','ws','%Y','%Y_%m*.nc'),0],
        ['ERA5','ws2',os.path.join(working_directory,'wind','ws2','%Y','%Y_%m*.nc'),0],
        ['ERA5','msl',os.path.join(working_directory,'pressure','%Y','%Y_%m*.nc'),0]
        ]
    # This script cycles through all the vars and puts all the data in one file
    cinp.driver(data_file,vars,start_yr = start_yr,end_yr = end_yr,lon = log,lat = lag,copts=copts);

    # Here we locad all the data into a direct dictionary to then be made into individaul FluxEngine
    # input files.
    direct = {}
    c = Dataset(data_file,'r')
    keys = list(c.variables.keys())
    keys.remove('latitude'); keys.remove('longitude'); keys.remove('time');
    print(keys)
    for key in keys:
        direct[key] = np.array(c.variables[key][:])
    # direct['model_fice'][:] = 0 # Temporary code snippet until ice fields are avaiable... DJF: Ice fields avaiable.
    c.close()

    fl.fluxengine_individual_netcdf(model_location,direct,log,lag,start_yr = start_yr,end_yr = end_yr)

    # Now we run FluxEngine. Instead of editing the config file for each model run, we change directory so the config
    # file can be run from anywhere. This limits the chance for mistakes when having loads of config files.
    # return_path = os.getcwd()
    # os.chdir(model_location)
    # returnCode, fe = fluxengine.run_fluxengine(flux_config, start_yr, end_yr, singleRun=False,verbose=False,processLayersOff=True);
    # os.chdir(return_path)

    # Now we load the data from the FluxEngine files
    flux = fl.load_flux_var(os.path.join(model_location,'flux'),'OF',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)
    ice = fl.load_flux_var(os.path.join(model_location,'flux'),'P1',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)
    dpco2 = fl.load_flux_var(os.path.join(model_location,'flux'),'dpCO2',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)
    schmidt = fl.load_flux_var(os.path.join(model_location,'flux'),'SC',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)
    sol = fl.load_flux_var(os.path.join(model_location,'flux'),'fnd_solubility',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)
    pgas = fl.load_flux_var(os.path.join(model_location,'flux'),'OAPC1',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)
    k = fl.load_flux_var(os.path.join(model_location,'flux'),'OK3',start_yr,end_yr,len(log),len(lag),(end_yr-start_yr+1)*12)

    flux = flux * (1-ice) # Flux needs to be multiplied by 1-ice as this part isn't done in FluxEngine
    flux = np.transpose(flux,(1,0,2)) # We transpose these as they are in (time, latitude, longitude) and we want (longitude, latitude, time).
    dpco2 = np.transpose(dpco2,(1,0,2))
    schmidt = np.transpose(schmidt,(1,0,2))
    sol = np.transpose(sol,(1,0,2))
    pgas = np.transpose(pgas,(1,0,2))
    k = np.transpose(k,(1,0,2))
    #
    # Now we can append all the data back to the orginial data file.
    c = Dataset(data_file,'a')
    keys = c.variables.keys()
    if 'model_flux' in keys:
        c.variables['model_flux'][:] = flux
    else:
        var_o = c.createVariable('model_flux','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = flux
    c.variables['model_flux'].Long_name = 'Model Air-sea CO2 flux'
    c.variables['model_flux'].Units = 'g C m-2 d-1'
    c.variables['model_flux'].direction = '-ve into the oceans'
    
    if 'model_dfCO2' in keys:
        c.variables['model_dfCO2'][:] = dpco2
    else:
        var_o = c.createVariable('model_dfCO2','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = dpco2
    c.variables['model_dfCO2'].Long_name = 'Model delta fCO2'
    c.variables['model_dfCO2'].Units = 'uatm'
    c.variables['model_dfCO2'].direction = '-ve indicates seawater is less than atmospheric fCO2'
    
    if 'schmidt' in keys:
        c.variables['schmidt'][:] = schmidt
    else:
        var_o = c.createVariable('schmidt','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = schmidt
    c.variables['schmidt'].Long_name = 'Schmidt number for air-sea CO2 flux calculation'
    c.variables['schmidt'].Units = 'unitless'
    c.variables['schmidt'].description = 'Calculated from model temperature'
    
    if 'solubility' in keys:
        c.variables['solubility'][:] = sol
    else:
        var_o = c.createVariable('solubility','f4',('longitude','latitude','time'),fill_value=np.nan)
        var_o[:] = sol

    c.variables['solubility'].Long_name = 'Solubility for air-sea CO2 flux calculation'
    c.variables['solubility'].Units = 'mol L-1 atm-1'
    c.variables['solubility'].description = 'Calculated from model temperature and salinity'
    
    if 'atm_fco2' in keys:
        c.variables['atm_fco2'][:] = pgas
    else:
        var_o = c.createVariable('atm_fco2','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = pgas
    c.variables['atm_fco2'].Long_name = 'Fugacity of CO2 in atmosphere'
    c.variables['atm_fco2'].Units = 'uatm'
    c.variables['atm_fco2'].dataset = 'xCO2atm (global average monthly) converted to fCO2atm with ERA5 sea level pressure'

    if 'k' in keys:
        c.variables['k'][:] = k
    else:
        var_o = c.createVariable('k','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = k
        
    c.variables['k'].Long_name = 'Gas transfer coefficient at the Schmidt number'
    c.variables['k'].Units = 'cm hr-1'
    c.variables['k'].description = 'Calculated from ERA5 winds using c = 0.271'
    c.variables['k'].formulation = '(660/Schmidt)^-0.5 * kw'

    c.variables['model_fice'].Long_name = 'Ice coverage'
    c.variables['model_fice'].dataset = 'Model ice'

    c.variables['ERA5_ws'].Long_name = 'Wind speed'
    c.variables['ERA5_ws'].Units = 'ms-1'
    c.variables['ERA5_ws'].dataset = 'ERA5 Hourly'
    
    c.variables['ERA5_ws2'].Long_name = 'Second moment wind speed'
    c.variables['ERA5_ws2'].Units = '(ms-1)^2'
    c.variables['ERA5_ws2'].dataset = 'ERA5 Hourly'

    c.variables['ERA5_msl'].Long_name = 'Sea level pressure'
    c.variables['ERA5_msl'].Units = 'Pa'
    c.variables['ERA5_msl'].dataset = 'ERA5 Monthly'

    c.variables['model_tos'].Long_name = 'Sea surface temperature'
    c.variables['model_tos'].Units = 'degC'
    c.variables['model_tos'].dataset = 'Model temperature'

    c.variables['model_sos'].Long_name = 'Sea surface salinity'
    c.variables['model_sos'].Units = 'psu'
    c.variables['model_sos'].dataset = 'Model salinity'

    c.variables['model_sfco2'].Long_name = 'Model surface ocean fCO2'
    c.variables['model_sfco2'].Units = 'uatm'

    c.variables['model_xco2'].Long_name = 'Atmospheric xCO2'
    c.variables['model_xco2'].Units = 'ppm'
    c.variables['model_xco2'].dataset = 'xCO2atm provide with each mode(global average monthly)'

    d = Dataset(area_file,'r')
    if 'area' in keys:
        c.variables['area'][:] = np.array(d.variables['area'][:])
    else:
        var_o = c.createVariable('area','f4',('longitude','latitude'),fill_value=np.nan,**copts)
        var_o[:] = np.array(d.variables['area'][:])
    c.variables['area'].Long_name = 'Total surface area of each grid cell'
    c.variables['area'].Units = 'm2'
    c.variables['area'].description = 'Calculated assuming the Earth is a oblate sphere with major and minor radius of 6378.137 km and 6356.7523 km respectively'
    c.variables['area'].comment = 'Multiply "area" and "mask_sfc" to get true area used in flux calculations.'
    
    if 'mask_sfc' in keys:
        c.variables['mask_sfc'][:] = np.array(d.variables['ocean_proportion'][:])
    else:
        var_o = c.createVariable('mask_sfc','f4',('longitude','latitude'),fill_value=np.nan,**copts)
        var_o[:] = np.array(d.variables['ocean_proportion'][:])
    c.variables['mask_sfc'].Long_name = 'Fractional coverage of ocean in each grid tile'
    c.variables['mask_sfc'].Units = ''
    c.variables['mask_sfc'].description = 'This is the ocean proportion mask calculated from ESA-CCI Land.'
    c.close()
    d.close()

## Merging in the data product fCO2sw and calculating their fluxes

In this section I run through for each model and add the data product fCO2sw data for each of the OSSE experiments (i.e OSSE1-7).
I use the already avaiable fields of solubility, atmospheric fco2 and k to calculate the fluxes instead of running FluxEngine for all the combinations.

Here we rely on the groups formatting the file names correctly, and if not this needs to be corrected manually.

In [6]:
gas_convert = 24/100 /1000 # Convert from hr-1 to day-1, cm to m, and L-1 to m-3

files = glob.glob(os.path.join(data_prod_directory,'*.nc')) # Find all the netCDF files in the output folder

# Probably a better way to do this, but it sets up an array of the year of each timestep so we can merge in different
# products onto a consistent grid.
t = []
yr=start_yr
mon=1
while yr<=end_yr:
    t.append(yr)
    mon=mon+1
    if mon==13:
        yr = yr+1
        mon=1
t = np.array(t)

#Now we merge in the data products and calcualte the fluxes
for file in files: # Cycle through the files in the output folder
    print(file)
    s = file.split(os.path.sep)[-1].split('_') # Split the filename by the default os filename seperator
    osse = s[0].split('-')[0] # The OSSE experiment nunber
    method = s[2] # The method
    model = s[3] # The model
    st_yr = int(s[-1].split('-')[0]) # Product start year
    en_yr = int(s[-1].split('-')[1].split('.')[0]) # Product end year
    
    sfco2_temp = np.zeros((len(log),len(lag),(end_yr-start_yr+1)*12)); sfco2_temp[:] = np.nan # Setup nan array 
    
    c = Dataset(file,'r') # Open the data product file
    try:
        sfco2 = c.variables['sfco2'][:] # Load the sfCO2 data
        try: # Try and apply the fill value if it's specified
            sfco2[sfco2 == c.variables['sfco2']._FillValue] = np.nan
        except:
            print('No fill value defined in file') # Else we just print
    except:
        print('No sfco2, trying fco2_reconstructed')
        sfco2 = c.variables['fco2_reconstructed'][:] # Load the sfCO2 data
        try: # Try and apply the fill value if it's specified
            sfco2[sfco2 == c.variables['fco2_reconstructed']._FillValue] = np.nan
        except:
            print('No fill value defined in file') # Else we just print
    c.close()
    
    f = np.where((t>=st_yr) & (t<=en_yr))[0] # Find where the product time period overlaps with the model array
    sfco2_temp[:,:,f] = np.transpose(np.roll(sfco2,180,axis=2),[2,1,0]) # Now we roll (0-360 to -180-180 and transpose to (lon, lat, time).

    d = Dataset(os.path.join(truth_directory,model.replace('-','_'),'OSSE_'+model.replace('-','_')+'_'+version+'.nc'),'a') # Load the model base file
    keys = d.variables.keys() # Get all the variable names
    # Save the data product fields into the file with the OSSE number at start
    if osse+'_'+method+'_sfco2' in keys:
        d.variables[osse+'_'+method+'_sfco2'][:] = sfco2_temp
    else:
        var_o = d.createVariable(osse+'_'+method+'_sfco2','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = sfco2_temp
    d.variables[osse+'_'+method+'_sfco2'].Units = 'uatm'

    #Loading the flux variables needed for the flux calculation
    atm = d.variables['atm_fco2'][:]
    k = d.variables['k'][:]
    sol = d.variables['solubility'][:]
    ice = d.variables['model_fice'][:]
    flux = k * gas_convert * 12.0107 * sol * (sfco2_temp - atm) * (1-ice) # Calculate the data product flux

    # Save the data product flux into file
    if osse+'_'+method+'_flux' in keys:
        d.variables[osse+'_'+method+'_flux'][:] = flux
    else:
        var_o = d.createVariable(osse+'_'+method+'_flux','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = flux
    d.variables[osse+'_'+method+'_flux'].Units = 'g C m-2 d-1'
    d.close()

D:/OSSE_Experiments\Output\OSSE3-2026_Base+Disc_NIES-ML3_FESOM2-REcoM_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE3-2026_Base+Disc_NIES-ML3_IPSL-NEMO-PISCES_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE3-2026_Base+Disc_NIES-ML3_MRI-ESM2_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE4-2026_Base+VOS_NIES-ML3_FESOM2-REcoM_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE4-2026_Base+VOS_NIES-ML3_IPSL-NEMO-PISCES_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE4-2026_Base+VOS_NIES-ML3_MRI-ESM2_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE5-2026_Base+RV_NIES-ML3_FESOM2-REcoM_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE5-2026_Base+RV_NIES-ML3_IPSL-NEMO-PISCES_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE5-2026_Base+RV_NIES-ML3_MRI-ESM2_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE6-2026_Base+All+USV_NIES-ML3_FESOM2-REcoM_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE6-2026_Base+All+USV_NIES-ML3_IPSL-NEMO-PISCES_1980-2024.nc
D:/OSSE_Experiments\Output\OSSE6-2026_Base+All+USV_NIES-ML3_MRI-ESM2_1980-2024.nc
D:/O

## Generate a consistent mask

In [7]:
d = Dataset(reccap_file,'r')
open_ocean = np.roll(np.transpose(np.array(d['open_ocean'])),180,axis=0)
d.close()

for model in list(models.keys()):
    print(model)
    data_file = os.path.join(truth_directory,model,'OSSE_'+model+'_'+version+'.nc')
    c = Dataset(data_file,'a')
    d = list(c.variables.keys()) # Get a list of the variables in the file
    lat = np.array(c.variables['latitude'])
    co = 0 # A counter so we know how many OSSE's are considered
    for i in d:
        if ('OSSE' in i) and ('sfco2' in i): # Only include when OSSE and sfco2 are included in the name
            print(i)
            data = np.array(c[i]) # Load the dataset
            if co == 0:
                counter = np.zeros((data.shape)) # If co is 0 then we need to make the counter array
            f = np.where(np.isnan(data) == 0) # Where the data files is not nan
            counter[f] = counter[f]+1 # Add one to these regions
            co = co+1 # Add one to our main counter
    cons = np.zeros((counter.shape)) # Make a new array
    cons[counter == co] = 1 # Whereever the counter array (2D) is equal to the counter number all methods have values and the mask is set to 1
    
    # Now save this as our global consistent mask
    if 'consistent_mask' in d:
        c.variables['consistent_mask'][:] = cons
    else:
        var_o = c.createVariable('consistent_mask','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = cons
    
    # Setup for our regional masks (North)
    cons_temp = np.copy(cons)
    f = np.where(lat < 30)[0]
    cons_temp[:,f,:] = 0

    if 'consistent_mask_north' in d:
        c.variables['consistent_mask_north'][:] = cons_temp
    else:
        var_o = c.createVariable('consistent_mask_north','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = cons_temp
    
    # Setup for our regional masks (Equator)
    cons_temp = np.copy(cons)
    f = np.where((lat < -30) | (lat>30) )[0]
    cons_temp[:,f,:] = 0

    if 'consistent_mask_equator' in d:
        c.variables['consistent_mask_equator'][:] = cons_temp
    else:
        var_o = c.createVariable('consistent_mask_equator','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = cons_temp

    # Setup for our regional masks (South)
    cons_temp = np.copy(cons)
    f = np.where(lat > -30)[0]
    cons_temp[:,f,:] = 0

    if 'consistent_mask_south' in d:
        c.variables['consistent_mask_south'][:] = cons_temp
    else:
        var_o = c.createVariable('consistent_mask_south','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = cons_temp
    
    # Setup for our regional masks (Southern Ocean using RECCAP2 mask)
    cons_temp = np.copy(cons)
    f = np.where(open_ocean!=5)
    cons_temp[f[0],f[1],:] = 0

    if 'consistent_mask_southernocean' in d:
        c.variables['consistent_mask_southernocean'][:] = cons_temp
    else:
        var_o = c.createVariable('consistent_mask_southernocean','f4',('longitude','latitude','time'),fill_value=np.nan,**copts)
        var_o[:] = cons_temp
    c.close()

FESOM2_REcoM
OSSE3_NIES-ML3_sfco2
OSSE4_NIES-ML3_sfco2
OSSE5_NIES-ML3_sfco2
OSSE6_NIES-ML3_sfco2
OSSE7_NIES-ML3_sfco2
OSSE1_NIES-ML3_sfco2
OSSE2_NIES-ML3_sfco2
OSSE6_UExP-FNN-U-v1_sfco2
OSSE1_UExP-FNN-U-v1_sfco2
OSSE2_UExP-FNN-U-v1_sfco2
OSSE3_UExP-FNN-U-v1_sfco2
OSSE4_UExP-FNN-U-v1_sfco2
OSSE5_UExP-FNN-U-v1_sfco2
OSSE1_LDEO_sfco2
OSSE2_LDEO_sfco2
OSSE3_LDEO_sfco2
OSSE4_LDEO_sfco2
OSSE5_LDEO_sfco2
OSSE6_LDEO_sfco2
OSSE7_LDEO_sfco2
IPSL_NEMO_PISCES
OSSE3_NIES-ML3_sfco2
OSSE4_NIES-ML3_sfco2
OSSE5_NIES-ML3_sfco2
OSSE6_NIES-ML3_sfco2
OSSE7_NIES-ML3_sfco2
OSSE1_NIES-ML3_sfco2
OSSE2_NIES-ML3_sfco2
OSSE6_UExP-FNN-U-v1_sfco2
OSSE1_UExP-FNN-U-v1_sfco2
OSSE2_UExP-FNN-U-v1_sfco2
OSSE3_UExP-FNN-U-v1_sfco2
OSSE4_UExP-FNN-U-v1_sfco2
OSSE5_UExP-FNN-U-v1_sfco2
OSSE1_LDEO_sfco2
OSSE2_LDEO_sfco2
OSSE3_LDEO_sfco2
OSSE4_LDEO_sfco2
OSSE5_LDEO_sfco2
OSSE6_LDEO_sfco2
OSSE7_LDEO_sfco2
MRI_ESM2
OSSE3_NIES-ML3_sfco2
OSSE4_NIES-ML3_sfco2
OSSE5_NIES-ML3_sfco2
OSSE6_NIES-ML3_sfco2
OSSE7_NIES-ML3_sfco2
OSSE1_NIES-M

## Calculating intergrated fluxes on a consistent geographical coverage



In [8]:
import fluxengine_driver as fl

masks = ['consistent_mask','consistent_mask_equator','consistent_mask_north','consistent_mask_south','consistent_mask_southernocean']
for model in list(models.keys()):
    output_loc = os.path.join(truth_directory,model,'intergrated_fluxes')
    du.makefolder(output_loc)
    print(model)
    data_file = os.path.join(truth_directory,model,'OSSE_'+model+'_'+version+'.nc')
    c = Dataset(data_file,'r')
    d = list(c.variables.keys())
    for i in d:
        if ('flux' in i):
            print(i)
            for j in masks:
                print(j)
                fl.calc_annual_flux(os.path.join(truth_directory,model),
                                    lon=log,
                                    lat=lag,
                                    start_yr=start_yr,
                                    end_yr=end_yr,
                                    flux_file = data_file,
                                    flux_var = i,
                                    ice_var = 'model_fice',
                                    save_file=os.path.join(output_loc,i+'_' + j+'.csv'),
                                    mask_file=data_file,
                                    mask_var = j,
                                    land_file = area_file)
                os.remove(os.path.join(output_loc,i+'_' + j+ '_monthly.csv'))

    c.close()

FESOM2_REcoM
model_flux
consistent_mask
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_equator
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_north
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_south
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_southernocean
1.0
(180, 360, 540)
(180, 360, 540)
OSSE3_NIES-ML3_flux
consistent_mask
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_equator
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_north
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_south
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_southernocean
1.0
(180, 360, 540)
(180, 360, 540)
OSSE4_NIES-ML3_flux
consistent_mask
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_equator
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_north
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_south
1.0
(180, 360, 540)
(180, 360, 540)
consistent_mask_southernocean
1.0
(180, 360, 540)
(180, 360, 540)
OSSE5_NIES-ML3_flux
consistent_mask
1.0
(180, 360, 540

## Some plotting

In [14]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.transforms
font = {'weight' : 'normal',
            'size'   : 12}
matplotlib.rc('font', **font)
cols = ['#332288','#44AA99','#882255','#DDCC77', '#117733', '#88CCEE','#999933','#CC6677']

plot_loc = os.path.join(working_directory,'plots')
du.makefolder(plot_loc)

masks = ['consistent_mask','consistent_mask_southernocean']
lims = [-3.5,0.5]
for model in list(models.keys()):
    output_loc = os.path.join(truth_directory,model,'intergrated_fluxes')
    g = glob.glob(os.path.join(output_loc,'OSSE*.csv'))
    methods = []
    for i in range(len(g)):
        s = g[i].split(os.path.sep)[-1].split('_')[1]
        if s not in methods:
            methods.append(s)
    methods = sorted(methods)
    print(methods)
    
    for mask in masks:
        m2 = np.zeros((len(methods),2))
        fig,ax = plt.subplots(int(np.ceil(len(methods)/3)),3,figsize=(15*int(np.ceil(len(methods)/3)),5))
        ax = ax.ravel()
        for j in range(len(methods)):
            m = np.zeros((7,2)); m[:] = np.nan
            # Shrink current axis by 20%
            # box = ax[j].get_position()
            # ax[j].set_position([box.x0, box.y0, box.width * 0.8, box.height])
            data = pd.read_table(os.path.join(output_loc,'model_flux_'+mask+'.csv'),sep=',')
            ax[j].plot(data['# Year'],data['Net air-sea CO2 flux (Pg C yr-1)'],'k-',label='Truth',linewidth=2)
            # int_files = glob.glob(os.path.join(output_loc,'OSSE*_'+methods[j]+'_flux_consistent.csv'))
            for i in range(7):
                file = os.path.join(output_loc,'OSSE' + str(i+1)+'_'+methods[j]+'_flux_' + mask + '.csv')
                osse = i+1
                # print(osse)
                try:
                    data = pd.read_table(file,sep=',')
                    ax[j].plot(data['# Year'],data['Net air-sea CO2 flux (Pg C yr-1)'],label='OSSE'+str(osse),color=cols[int(osse)-1])
                    m[i,:] = [np.min(data['Net air-sea CO2 flux (Pg C yr-1)']), np.max(data['Net air-sea CO2 flux (Pg C yr-1)'])]
                except:
                    print('No file')    
            # Put a legend to the right of the current axis
            # ax[j].legend(loc='center left', bbox_to_anchor=(1, 0.5))
            ax[j].legend(fontsize=10)
            ax[j].set_title(methods[j])
            ax[j].grid()
            ax[j].set_xlim([start_yr,end_yr])
            
            ax[j].set_ylabel('Net air-sea CO$_2$ flux (Pg C yr$^{-1}$) \n (-ve into ocean; no riverine correction)')
            ax[j].set_xlabel('Year')
            #print(m)
            m2[j,:] = [np.nanmin(m[:,0]),np.nanmax(m[:,1])]
            #print(m2)
        plt.suptitle(model + ' ' + mask + '\n('+datetime.datetime.now().strftime('%Y-%m-%d %H:%M')+')')
        #print(m2)
        
        lims = [np.min(m2[:,0])-0.1,np.max(m2[:,1])+0.1]
        #print(lims)
        for j in range(len(methods)):
            ax[j].set_ylim(lims)
        plt.tight_layout()
        fig.savefig(os.path.join(plot_loc,model+'_'+mask+'_'+version+'.png'),dpi=150)
        plt.close(fig)
        

['LDEO', 'NIES-ML3', 'UExP-FNN-U-v1']
No file
No file
['LDEO', 'NIES-ML3', 'UExP-FNN-U-v1']
No file
No file
['LDEO', 'NIES-ML3', 'UExP-FNN-U-v1']
No file
No file
